# Drone detection training (YOLOv11)

Clean a single-class drone dataset and train YOLOv11 on Colab.
Update zip_path if your data lives somewhere else. Run the cells in order;
each one explains why it exists so you can change things confidently.


In [ ]:
# Setup: mount Google Drive in Colab and import the libraries we use.
# If you run locally, comment out the drive.mount line and adjust paths below.
from google.colab import drive  # safe to leave in Colab
import os
import zipfile
import shutil
import glob

drive.mount('/content/drive')


In [ ]:
# Unzip the labeled dataset from Drive into /content.
zip_path = '/content/drive/My Drive/Drone detection dataset.zip'  # change to your zip
extract_path = '/content/drone_detection_dataset'

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(f'Dataset extracted to {extract_path}')


In [ ]:
# Paths to the YOLO split folders we expect after unzip.
folders = ['train', 'valid', 'test']
base_path = extract_path


In [ ]:
# Keep only the "drone" class from the labels.
# Original files may have classes 0/1/2; we keep class 2 and relabel it to 0.
for folder in folders:
    labels_dir = os.path.join(base_path, folder, 'labels')
    label_files = glob.glob(os.path.join(labels_dir, '*.txt'))

    for file in label_files:
        with open(file, 'r') as f:
            lines = f.readlines()

        drone_only = [ln for ln in lines if ln.startswith('2 ')]

        if not drone_only:
            # Leave an empty file so YOLO knows the image exists but has no drone.
            open(file, 'w').close()
            continue

        remapped = ['0' + ln[1:] for ln in drone_only]  # change class id 2 -> 0
        with open(file, 'w') as f:
            f.writelines(remapped)

print('Labels cleaned: kept class 2, remapped it to class 0.')


In [ ]:
yaml_content = "\n".join([
    f"train: {base_path}/train/images",
    f"val: {base_path}/valid/images",
    f"test: {base_path}/test/images",
    "names:",
    "  0: drone",
    "nc: 1",
    "",
])

with open(os.path.join(base_path, 'data.yaml'), 'w') as f:
    f.write(yaml_content)

print(f"data.yaml saved to {os.path.join(base_path, 'data.yaml')}")


In [ ]:
# Optional: save the cleaned dataset back to Drive so you don't have to clean again.
save_back_to_drive = False  # set to True to enable the copy
drive_target = '/content/drive/My Drive/drone_detection_dataset'

if save_back_to_drive:
    shutil.copytree(base_path, drive_target, dirs_exist_ok=True)
    print(f'Copied cleaned dataset to {drive_target}')


In [ ]:
# Install the Ultralytics package (YOLOv11).
!pip install -q ultralytics --upgrade


In [ ]:
from ultralytics import YOLO


In [ ]:
# Optional: check which GPU Colab gave you.
!nvidia-smi


In [ ]:
# Train from scratch with the small YOLOv11 nano model.
data_yaml = os.path.join(base_path, 'data.yaml')
model = YOLO('yolo11n.pt')  # nano = fastest; switch to yolo11s.pt for more accuracy

results = model.train(
    data=data_yaml,
    imgsz=320,           # 320x320 images are quick on the free T4
    batch=16,            # fits in free Colab RAM; reduce if you hit OOM
    epochs=30,
    device=0,            # 0 = first GPU
    project='/content/drive/MyDrive/drone_detection_run2',
    name='train_fast_320',
    exist_ok=True,
    workers=2,
    patience=10,
    cache=False,
    close_mosaic=10
)


In [ ]:
# Resume training from the last checkpoint if you stopped the notebook.
run_dir = '/content/drive/MyDrive/drone_detection_run2/train_fast_320'
last_weights = os.path.join(run_dir, 'weights', 'last.pt')
model = YOLO(last_weights)

results = model.train(
    data=data_yaml,
    imgsz=320,
    batch=16,
    epochs=30,  # total target epochs; YOLO will keep the previous progress
    device=0,
    project='/content/drive/MyDrive/drone_detection_run2',
    name='train_fast_320',
    exist_ok=True,
    workers=2,
    patience=10,
    cache=False,
    close_mosaic=10,
    resume=True
)
